In [40]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.impute import SimpleImputer

# Feature Selection

In [2]:
merged_df = pd.read_csv("../data/processed/merged_clean.csv")

### Layer 1: Missing value threshold (dropping columns with too much missing data)

In [ ]:
missing_percentage = merged_df.isnull().sum()/len(merged_df)
missing_percentage = missing_percentage*100
less_than_40 = missing_percentage[missing_percentage<=40]
print(len(less_than_40))


86


In [ ]:
merged_df = merged_df[list(less_than_40.index)]
print(merged_df.shape)
print(merged_df.columns)

### Layer 2 — Domain relevance (drop columns unrelated to diabetes risk)

In [32]:
columns = ["IMONTH", "IDAY", "IYEAR", "FMONTH", "IDATE", "SEQNO", "QSTVER", "QSTLANG", "MSCODE", "DISPCODE", "CTELENUM", "_STSTR", "_RAWRAKE", "_STRWT", "_PSU", "_WT2RAKE", "_LLCPWT", "_RFSEAT2", "_RFSEAT3", "SEATBELT", "NUMADULT", "NUMMEN", "NUMWOMEN", "NUMHHOL2", "CPDEMO1"]
merged_df = merged_df.drop(columns=columns)

In [35]:
cols_to_keep = ["Diabetes_binary", "PHYSHLTH", "_SMOKER3", "GENHLTH", "CHECKUP1", "DROCDY3_", "_RFBING5", "_DRDXAR1", "PNEUVAC3", "_AIDTST3", "HLTHPLN1", "MENTHLTH", "_AGE_G", "QLACTLM2", "CHCOCNCR", "_RFHLTH", "HAVARTH3", "HIVTST6", "MEDCOST", "CHCSCNCR", "_AGE65YR", "EXERANY2", "SEX", "DRNKANY5", "_HCVU651", "_AGEG5YR", "_BMI5", "ALCDAY5", "ADDEPEV2", "PERSDOC2", "CVDSTRK3", "_TOTINDA", "CHCKIDNY", "_CHLDCNT", "INCOME2", "EDUCA", "CVDCRHD4", "CVDINFR4","_ASTHMS1"]

In [37]:
df = merged_df[cols_to_keep]
print(df.shape)
print(df.isnull().sum())

(2374814, 39)
Diabetes_binary         0
PHYSHLTH               16
_SMOKER3                0
GENHLTH                19
CHECKUP1                4
DROCDY3_                8
_RFBING5                0
_DRDXAR1            13077
PNEUVAC3           141243
_AIDTST3           165289
HLTHPLN1                2
MENTHLTH               10
_AGE_G                  0
QLACTLM2            60300
CHCOCNCR                3
_RFHLTH                 0
HAVARTH3                6
HIVTST6            165289
MEDCOST                 4
CHCSCNCR                9
_AGE65YR                0
EXERANY2            87220
SEX                     0
DRNKANY5                0
_HCVU651                0
_AGEG5YR                0
_BMI5              145961
ALCDAY5             83813
ADDEPEV2               12
PERSDOC2                2
CVDSTRK3                0
_TOTINDA                0
CHCKIDNY                6
_CHLDCNT                0
INCOME2             10128
EDUCA                3082
CVDCRHD4                1
CVDINFR4                

## Layer 3 : Dropping or Imputing Columns based on Redundancy

In [38]:
neg_missing = ["PHYSHLTH", "GENHLTH", "CHECKUP1", "DROCDY3_", "HLTHPLN1", "MENTHLTH", "CHCOCNCR", "HAVARTH3", "MEDCOST", "CHCSCNCR", "CVDCRHD4", "CVDINFR4"]
df = df.dropna(subset=neg_missing)

In [41]:
imputer = SimpleImputer(strategy='most_frequent')
impute = pd.DataFrame(df[["_AIDTST3", "HIVTST6", "PNEUVAC3", "EXERANY2", "QLACTLM2", "_DRDXAR1"]])
df[["_AIDTST3", "HIVTST6", "PNEUVAC3", "EXERANY2", "QLACTLM2", "_DRDXAR1"]] = imputer.fit_transform(impute)
print(df.isnull().sum())
print(df.shape)

Diabetes_binary         0
PHYSHLTH                0
_SMOKER3                0
GENHLTH                 0
CHECKUP1                0
DROCDY3_                0
_RFBING5                0
_DRDXAR1                0
PNEUVAC3                0
_AIDTST3                0
HLTHPLN1                0
MENTHLTH                0
_AGE_G                  0
QLACTLM2                0
CHCOCNCR                0
_RFHLTH                 0
HAVARTH3                0
HIVTST6                 0
MEDCOST                 0
CHCSCNCR                0
_AGE65YR                0
EXERANY2                0
SEX                     0
DRNKANY5                0
_HCVU651                0
_AGEG5YR                0
_BMI5              145947
ALCDAY5             83798
ADDEPEV2               12
PERSDOC2                0
CVDSTRK3                0
_TOTINDA                0
CHCKIDNY                6
_CHLDCNT                0
INCOME2             10119
EDUCA                3074
CVDCRHD4                0
CVDINFR4                0
_ASTHMS1    

In [43]:
df.to_csv("../data/processed/features_selected.csv", index=False)

## Documentation for Feature Selection

1) Started with 157 columns and ended with 39 after feature selection

2) I used a 3 Layer Filtering Approach
    - Layer 1: Missing value threshold (dropping columns with too much missing data)
    - Layer 2: Domain relevance (drop columns unrelated to diabetes risk)
    - Layer 3: Dropping or Imputing Columns based on Redundancy

3) Columns Dropped and Why
    - Administrative/Survey Metadata: IMONTH, IDAY, IYEAR, FMONTH, IDATE, SEQNO, 
      QSTVER, QSTLANG, MSCODE, DISPCODE, CTELENUM, _STSTR, _RAWRAKE, _STRWT, 
      _PSU, _WT2RAKE, _LLCPWT, SEATBELT, NUMADULT, NUMMEN, NUMWOMEN, NUMHHOL2, 
      CPDEMO1 — these are survey weights and interview metadata, not health features
    - Redundant BMI columns: HEIGHT3, HTIN4, WTKG3, WEIGHT2 — raw height and weight 
      are already captured in _BMI5 (continuous BMI). Keeping both would cause 
      multicollinearity
    - Redundant Smoking columns: SMOKE100, _RFSMOK3, USENOW3 — kept _SMOKER3 as it 
      captures full smoking status (current/former/never) in one column
    - Redundant Asthma columns: ASTHMA3, _CASTHM1, _LTASTH1 — kept _ASTHMS1 as it 
      captures current asthma status most completely
    - Redundant BMI encoding: _BMI5CAT, _RFBMI5 — kept _BMI5 (continuous) as it 
      carries more information than category or binary versions
    - Irrelevant to diabetes risk: RENTHOM1, CHILDREN, _INCOMG, VETERAN3, USEEQUIP 
      — no meaningful clinical link to diabetes
    - DIABETE3 — dropped to prevent data leakage as it is the raw version of our 
      target variable Diabetes_binary

4) Missing Value Strategy
    - Group A (Negligible missing, <1000 rows): PHYSHLTH, GENHLTH, CHECKUP1, 
      DROCDY3_, HLTHPLN1, MENTHLTH, CHCOCNCR, HAVARTH3, MEDCOST, CHCSCNCR, 
      CVDCRHD4, CVDINFR4 — dropped rows as the loss is negligible (<0.05% of 
      2.37M rows)
    - Group B (Significant missing): _AIDTST3, HIVTST6, PNEUVAC3, EXERANY2, 
      QLACTLM2, _DRDXAR1 — imputed using SimpleImputer with strategy='most_frequent' 
      (mode) to prevent data loss. Dropping these rows would have removed 60K-165K 
      rows which is too significant to ignore

5) Sentinel Values Handled
    - DIABETE3 values 7.0 (Don't know) and 9.0 (Refused) were replaced with NaN 
      and dropped — total loss of 5,233 rows (~0.17% of data)

6) Final Class Distribution after Target Variable Mapping
    - Class 0 (Not Diabetic): 2,010,686 rows (84%)
    - Class 1 (Diabetic/Pre-Diabetic): 364,128 rows (16%)
    - This is a significant class imbalance and will be handled in Week 4 using 
      SMOTE, class_weight='balanced', and threshold tuning

7) Layer 3 (Statistical Correlation Analysis) is intentionally deferred to EDA 
   where a correlation heatmap will guide further feature removal rather than 
   doing it blindly 

## Post-Selection Update (Added During Explainability Phase)

PNEUVAC3 was included in the initial feature selection based on its 
correlation with the target variable (r=0.19, ranked 4th).

However, SHAP analysis in Week 6 confirmed this correlation reflects 
**reverse causality** — diabetic patients are prescribed pneumonia 
vaccines as a consequence of their diagnosis, not as a risk factor.

**PNEUVAC3 was removed before final model deployment.**

This highlights an important limitation of purely statistical feature 
selection — correlation-based methods cannot detect causal direction. 
Domain knowledge and explainability tools (SHAP) are essential 
complements to statistical filtering in healthcare ML.

Final feature count: 34 (reduced from 35 after SHAP-informed removal)